# Databricks/PySpark dla Data Analyst/Engineer — Moduł 13: MLlib -- ocena modeli i strojenie

W Module 12 zbudowaliśmy pierwsze modele i oceniliśmy je jedną miarą (RMSE, AUC) na
jednym podziale danych. Ten moduł pokazuje, jak robić to solidniej: dlaczego jeden
podział train/test to za mało, jak automatycznie przeszukać wiele konfiguracji
hiperparametrów, i jak wybrać najlepszy model bez ręcznego zgadywania.

## Spis treści
1. [Train/test/validation -- dlaczego jeden podział nie wystarcza](#sec1)
2. [Evaluatory dokładniej: metryki regresji i klasyfikacji](#sec2)
3. [ParamGridBuilder: siatka hiperparametrów](#sec3)
4. [CrossValidator: walidacja krzyżowa](#sec4)
5. [Podsumowanie i ćwiczenia](#sec5)


Uruchamiamy (albo pobieramy już istniejącą) `SparkSession` — punkt wejścia do wszystkiego w Sparku. W Databricks `spark` jest już gotowe automatycznie w każdym notebooku; ten sam kod działa też lokalnie.

In [ ]:
from pyspark.sql import SparkSession

# Dodatkowe opcje JVM (--add-opens) sa potrzebne, gdy uruchamiasz Sparka lokalnie na
# Javie 17+ i chcesz uzywac pandas UDF / Apache Arrow (Modul 8) -- nowsze JDK domyslnie
# blokuja refleksyjny dostep do pamieci poza stosem, ktorego potrzebuje Arrow. W
# Databricks nie trzeba tego ustawiac recznie -- platforma robi to za Ciebie.
_opcje_jvm = (
    "--add-opens=java.base/java.nio=ALL-UNNAMED "
    "--add-opens=java.base/java.lang=ALL-UNNAMED "
    "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED"
)

spark = (
    SparkSession.builder.appName("kurs_spark")
    .config("spark.driver.extraJavaOptions", _opcje_jvm)
    .config("spark.executor.extraJavaOptions", _opcje_jvm)
    .getOrCreate()
)
spark


Wczytujemy nasz wspólny zestaw danych — dane sklepu internetowego (`klienci`, `produkty`, `pracownicy`, `zamowienia`, `pozycje_zamowien`), dokładnie ten sam schemat co w kursie SQL, ale tym razem jako DataFrame'y Sparka.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)

# --- klienci -----------------------------------------------------------
n_klienci = 300
miasta = ["Warszawa", "Krakow", "Gdansk", "Wroclaw", "Poznan"]
segmenty = ["Nowy", "Standardowy", "Premium"]
prawdop_segmentu = [0.35, 0.45, 0.20]

klienci_pd = pd.DataFrame({
    "klient_id": np.arange(1, n_klienci + 1),
    "miasto": rng.choice(miasta, size=n_klienci, p=[0.30, 0.25, 0.15, 0.15, 0.15]),
    "segment": rng.choice(segmenty, size=n_klienci, p=prawdop_segmentu),
    "dni_od_rejestracji": rng.integers(1, 900, size=n_klienci),
})

# --- produkty ------------------------------------------------------------
n_produkty = 40
kategorie = ["Elektronika", "Odziez", "Dom", "Sport", "Ksiazki"]
srednie_ceny = {"Elektronika": 1200, "Odziez": 150, "Dom": 250, "Sport": 300, "Ksiazki": 50}

kategorie_produktow = rng.choice(kategorie, size=n_produkty)
produkty_pd = pd.DataFrame({
    "produkt_id": np.arange(1, n_produkty + 1),
    "kategoria": kategorie_produktow,
    "cena": [
        round(max(5.0, rng.normal(srednie_ceny[k], srednie_ceny[k] * 0.3)), 2)
        for k in kategorie_produktow
    ],
})

# --- pracownicy ----------------------------------------------------------
n_pracownicy = 12
regiony = ["Polnoc", "Poludnie", "Wschod", "Zachod"]
pracownicy_pd = pd.DataFrame({
    "pracownik_id": np.arange(1, n_pracownicy + 1),
    "region": rng.choice(regiony, size=n_pracownicy),
    "dzial": rng.choice(["Sprzedaz", "Obsluga"], size=n_pracownicy, p=[0.6, 0.4]),
})

# --- zamowienia ------------------------------------------------------------
n_zamowienia = 3000
klient_id_zamowien = rng.choice(klienci_pd["klient_id"], size=n_zamowienia)
pracownik_id_zamowien = rng.choice(pracownicy_pd["pracownik_id"], size=n_zamowienia)

segment_mapa = klienci_pd.set_index("klient_id")["segment"]
segment_zamowien = klient_id_zamowien.astype(object)
segment_zamowien = pd.Series(klient_id_zamowien).map(segment_mapa).values

# prawdopodobienstwo anulowania zalezy od segmentu klienta (celowo wbudowany efekt)
prawdop_anulowania = np.where(
    segment_zamowien == "Nowy", 0.22,
    np.where(segment_zamowien == "Standardowy", 0.10, 0.04),
)
anulowane = rng.binomial(1, prawdop_anulowania)
status_zamowien = np.where(
    anulowane == 1, "Anulowane",
    rng.choice(["Zlozone", "Wyslane", "Dostarczone"], size=n_zamowienia, p=[0.2, 0.3, 0.5]),
)

data_bazowa = pd.Timestamp("2024-01-01")
dni_offset = rng.integers(0, 730, size=n_zamowienia)

zamowienia_pd = pd.DataFrame({
    "zamowienie_id": np.arange(1, n_zamowienia + 1),
    "klient_id": klient_id_zamowien,
    "pracownik_id": pracownik_id_zamowien,
    "data_zamowienia": [data_bazowa + pd.Timedelta(days=int(d)) for d in dni_offset],
    "status": status_zamowien,
})

# --- pozycje_zamowien ------------------------------------------------------
wiersze_pozycji = []
pozycja_id = 1
cena_produktu = produkty_pd.set_index("produkt_id")["cena"]

for zamowienie_id in zamowienia_pd["zamowienie_id"]:
    liczba_pozycji = rng.integers(1, 5)
    wybrane_produkty = rng.choice(produkty_pd["produkt_id"], size=liczba_pozycji, replace=False)
    for produkt_id in wybrane_produkty:
        ilosc = int(rng.integers(1, 4))
        rabat = rng.choice([0.0, 0.05, 0.10, 0.15], p=[0.55, 0.20, 0.15, 0.10])
        wiersze_pozycji.append({
            "pozycja_id": pozycja_id,
            "zamowienie_id": int(zamowienie_id),
            "produkt_id": int(produkt_id),
            "ilosc": ilosc,
            "cena_jednostkowa": float(cena_produktu[produkt_id]),
            "rabat": float(rabat),
        })
        pozycja_id += 1

pozycje_zamowien_pd = pd.DataFrame(wiersze_pozycji)

# --- konwersja na DataFrame'y Sparka ---------------------------------------
klienci = spark.createDataFrame(klienci_pd)
produkty = spark.createDataFrame(produkty_pd)
pracownicy = spark.createDataFrame(pracownicy_pd)
zamowienia = spark.createDataFrame(zamowienia_pd)
pozycje_zamowien = spark.createDataFrame(pozycje_zamowien_pd)

klienci.show(5)


> ⚡ **spark.sql.shuffle.partitions -- domyślne 200 to za dużo dla małych danych**
>
> `CrossValidator` w sekcji 4 trenuje dziesiątki modeli pod rząd -- na naszym MAŁYM zbiorze domyślne 200 partycji przy każdym shufflu (Moduł 9) to czysty narzut planowania zadań, bez żadnej korzyści z równoległości. Obniżamy tę wartość na czas tego notatnika -- w prawdziwej pracy z dużymi danymi zostawiasz wartość domyślną albo dobierasz ją do rozmiaru klastra.

In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", "8")


In [ ]:
from pyspark.sql import functions as F

zamowienia_cechy = (
    zamowienia.join(klienci, "klient_id")
    .join(pozycje_zamowien, "zamowienie_id")
    .groupBy("zamowienie_id", "klient_id", "segment", "status")
    .agg(
        F.count("*").alias("liczba_pozycji"),
        F.avg("cena_jednostkowa").alias("srednia_cena"),
        F.sum(F.col("ilosc") * F.col("cena_jednostkowa") * (1 - F.col("rabat"))).alias("wartosc"),
    )
    .withColumn("czy_anulowane", (F.col("status") == "Anulowane").cast("double"))
)
zamowienia_cechy.show(5)


<a id="sec1"></a>
## 1. Train/test/validation -- dlaczego jeden podział nie wystarcza

W Module 12 dzieliliśmy dane na TYLKO dwie części: trening i test. To działa do prostej
demonstracji, ale ma pułapkę: jeśli używasz zbioru TESTOWEGO wielokrotnie, żeby
porównywać różne konfiguracje modelu i wybrać najlepszą, w praktyce zaczynasz
"dopasowywać się" do tego konkretnego zbioru testowego -- podobnie jak przy wielokrotnym
poprawianiu odpowiedzi na widok tego samego testu.

In [ ]:
# Trojpodzial: trening (do uczenia), walidacja (do wyboru hiperparametrow),
# test (jedna, finalna ocena -- dotykana tylko raz, na samym koncu)
trening, walidacja, test = zamowienia_cechy.randomSplit([0.6, 0.2, 0.2], seed=42)
print(f"Trening: {trening.count()}, walidacja: {walidacja.count()}, test: {test.count()}")


> ⚠️ **Zbiór testowy dotykasz TYLKO RAZ, na samym końcu**
>
> Zasada, którą łatwo złamać przez nieuwagę: zbiór WALIDACYJNY służy do porównywania konfiguracji i wybierania najlepszej (możesz go używać wielokrotnie). Zbiór TESTOWY służy WYŁĄCZNIE do jednej, ostatecznej oceny wybranego modelu -- jeśli zaczniesz na jego podstawie poprawiać model, przestaje on być wiarygodną, niezależną miarą jakości. `CrossValidator` (sekcja 4) automatyzuje podział trening/walidacja za Ciebie -- Ty i tak zawsze trzymasz osobny zbiór testowy z boku.

<a id="sec2"></a>
## 2. Evaluatory dokładniej: metryki regresji i klasyfikacji

`RegressionEvaluator` i `BinaryClassificationEvaluator` (Moduł 12) obsługują WIELE
metryk, nie tylko tę domyślną -- zmienioną przez `metricName` albo `setMetricName()`.

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

assembler = VectorAssembler(inputCols=["liczba_pozycji", "srednia_cena"], outputCol="features")
model_regresji = LinearRegression(featuresCol="features", labelCol="wartosc")

wytrenowany = assembler.transform(trening)
model = model_regresji.fit(wytrenowany)
predykcje = model.transform(assembler.transform(test))

for metryka in ["rmse", "mse", "mae", "r2"]:
    ewaluator = RegressionEvaluator(labelCol="wartosc", predictionCol="prediction", metricName=metryka)
    print(f"{metryka.upper()}: {ewaluator.evaluate(predykcje):.3f}")


In [ ]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

model_logistyczny = LogisticRegression(featuresCol="features", labelCol="czy_anulowane")
model_log = model_logistyczny.fit(assembler.transform(trening))
predykcje_log = model_log.transform(assembler.transform(test))

for metryka in ["areaUnderROC", "areaUnderPR"]:
    ewaluator = BinaryClassificationEvaluator(
        labelCol="czy_anulowane", rawPredictionCol="rawPrediction", metricName=metryka
    )
    print(f"{metryka}: {ewaluator.evaluate(predykcje_log):.3f}")

for metryka in ["accuracy", "f1", "weightedPrecision", "weightedRecall"]:
    ewaluator = MulticlassClassificationEvaluator(
        labelCol="czy_anulowane", predictionCol="prediction", metricName=metryka
    )
    print(f"{metryka}: {ewaluator.evaluate(predykcje_log):.3f}")


> ⚡ **accuracy bywa mylące przy niezbalansowanych klasach**
>
> W naszych danych zdecydowana większość zamówień NIE jest anulowana -- model, który ZAWSZE przewiduje 'nie anulowane', osiągnie wysoki `accuracy`, mimo że jest bezużyteczny. Dokładnie ten sam problem omawialiśmy przy regresji logistycznej w Module 13 kursu Statystyki -- dlatego `areaUnderROC`/`areaUnderPR` (nieczułe na nierównowagę klas) są tu bardziej wiarygodnymi metrykami niż samo `accuracy`.

<a id="sec3"></a>
## 3. ParamGridBuilder: siatka hiperparametrów

Zamiast RĘCZNIE próbować różnych wartości hiperparametrów (np. `regParam` -- siła
regularyzacji), `ParamGridBuilder` buduje SIATKĘ wszystkich kombinacji do
automatycznego przetestowania.

In [ ]:
from pyspark.ml.tuning import ParamGridBuilder

siatka = (
    ParamGridBuilder()
    .addGrid(model_regresji.regParam, [0.0, 0.1, 0.5, 1.0])
    .addGrid(model_regresji.elasticNetParam, [0.0, 0.5, 1.0])
    .build()
)
print(f"Liczba kombinacji do przetestowania: {len(siatka)}")


> 🧱 **regParam i elasticNetParam -- przypomnienie z kursu Statystyki**
>
> `regParam` kontroluje SIŁĘ regularyzacji (0 = brak, większe wartości = mocniejsze karanie dużych współczynników, przeciwdziałanie przeuczeniu). `elasticNetParam` miesza dwa typy regularyzacji: 0 to czysty Ridge (L2), 1 to czysty Lasso (L1) -- te same koncepcje, które można spotkać przy regresji regularyzowanej w `sklearn`/`statsmodels`.

<a id="sec4"></a>
## 4. CrossValidator: walidacja krzyżowa

`CrossValidator` łączy `ParamGridBuilder` z **k-krotną walidacją krzyżową**: dla KAŻDEJ
kombinacji hiperparametrów, dzieli dane treningowe na `k` części, trenuje `k` razy
(za każdym razem na innej kombinacji części jako trening/walidacja), uśrednia wynik, i
na końcu wybiera KOMBINACJĘ o najlepszym średnim wyniku.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.tuning import CrossValidator

pipeline = Pipeline(stages=[assembler, model_regresji])

siatka_pipeline = (
    ParamGridBuilder()
    .addGrid(model_regresji.regParam, [0.0, 0.1, 0.5, 1.0])
    .addGrid(model_regresji.elasticNetParam, [0.0, 0.5, 1.0])
    .build()
)

ewaluator_cv = RegressionEvaluator(labelCol="wartosc", predictionCol="prediction", metricName="rmse")

cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=siatka_pipeline,
    evaluator=ewaluator_cv,
    numFolds=3,
    seed=42,
)

cv_model = cv.fit(trening.union(walidacja.select(trening.columns)))
print(f"Sredni RMSE dla kazdej kombinacji: {[round(m, 2) for m in cv_model.avgMetrics]}")


In [ ]:
# Najlepszy model wybrany automatycznie -- ocena na NIETKNIETYM dotychczas zbiorze testowym
najlepszy_model = cv_model.bestModel
predykcje_najlepszego = najlepszy_model.transform(test)

rmse_najlepszego = ewaluator_cv.evaluate(predykcje_najlepszego)
print(f"RMSE najlepszego modelu na zbiorze testowym: {rmse_najlepszego:.2f}")

najlepszy_regresja = najlepszy_model.stages[-1]
print(f"Najlepszy regParam: {najlepszy_regresja.getRegParam()}")
print(f"Najlepszy elasticNetParam: {najlepszy_regresja.getElasticNetParam()}")


> ⚠️ **CrossValidator jest kosztowny obliczeniowo -- liczba kombinacji razy numFolds**
>
> Nasza siatka miała 4 x 3 = 12 kombinacji, przy `numFolds=3` -- to 36 pełnych treningów modelu. Na dużych danych i większych siatkach ten koszt rośnie szybko. `TrainValidationSplit` (siostrzana klasa `CrossValidator`, niewymieniona tu szczegółowo) robi TYLKO jeden podział trening/walidacja zamiast `k`-krotnego -- szybszy, ale mniej stabilny wynik. Wybór między nimi to kompromis między czasem obliczeń a wiarygodnością oceny.

<a id="sec5"></a>
## 5. Podsumowanie i ćwiczenia

W tym module poznaliśmy:
- podział train/validation/test -- i dlaczego zbiór testowy dotykamy tylko raz,
- pełny zestaw metryk `RegressionEvaluator` (rmse/mse/mae/r2) i `BinaryClassificationEvaluator`/`MulticlassClassificationEvaluator` (AUC, accuracy, F1, precyzja, recall),
- `ParamGridBuilder` -- budowanie siatki hiperparametrów do automatycznego przetestowania,
- `CrossValidator` -- k-krotna walidacja krzyżowa i automatyczny wybór najlepszej konfiguracji.

### 📝 Ćwiczenie 1: Trójpodział danych dla produktów

Podziel `produkty` na trening/walidację/test w proporcji 60/20/20 (`seed=1`) i wypisz liczbę wierszy w każdej części.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
tr, wal, te = produkty.randomSplit([0.6, 0.2, 0.2], seed=1)
print(f"Trening: {tr.count()}, walidacja: {wal.count()}, test: {te.count()}")
```

</details>

### 📝 Ćwiczenie 2: Wszystkie metryki regresji naraz

Dla modelu regresji z sekcji 2, wypisz WSZYSTKIE cztery metryki (rmse, mse, mae, r2) w jednej pętli `for`.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
for metryka in ["rmse", "mse", "mae", "r2"]:
    ew = RegressionEvaluator(labelCol="wartosc", predictionCol="prediction", metricName=metryka)
    print(f"{metryka}: {ew.evaluate(predykcje):.3f}")
```

</details>

### 📝 Ćwiczenie 3: Prosta siatka hiperparametrów

Zbuduj `ParamGridBuilder` testujący TYLKO `regParam` w wartościach `[0.0, 0.3, 0.6]` (bez `elasticNetParam`) i wypisz liczbę kombinacji.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
from pyspark.ml.tuning import ParamGridBuilder

siatka = ParamGridBuilder().addGrid(model_regresji.regParam, [0.0, 0.3, 0.6]).build()
print(f"Liczba kombinacji: {len(siatka)}")
```

</details>

### 📝 Ćwiczenie 4: CrossValidator dla regresji logistycznej

Zbuduj `CrossValidator` (numFolds=3) dla `Pipeline` z regresją logistyczną (cecha: `liczba_pozycji`, etykieta: `czy_anulowane`), testując `regParam` w `[0.0, 0.1, 0.5]`, i wypisz najlepszy `regParam`.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import BinaryClassificationEvaluator

assembler_prosty = VectorAssembler(inputCols=["liczba_pozycji"], outputCol="features")
model_log_prosty = LogisticRegression(featuresCol="features", labelCol="czy_anulowane")
pipeline_prosty = Pipeline(stages=[assembler_prosty, model_log_prosty])

siatka = ParamGridBuilder().addGrid(model_log_prosty.regParam, [0.0, 0.1, 0.5]).build()
ewaluator = BinaryClassificationEvaluator(
    labelCol="czy_anulowane", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)

cv = CrossValidator(
    estimator=pipeline_prosty, estimatorParamMaps=siatka, evaluator=ewaluator, numFolds=3, seed=1
)
cv_model = cv.fit(zamowienia_cechy)
print(f"Najlepszy regParam: {cv_model.bestModel.stages[-1].getRegParam()}")
```

</details>

> 🔥 **Wyzwanie: pełne porównanie modelu domyślnego i modelu ze strojeniem**
>
> Zbuduj DWA modele regresji liniowej przewidujące `wartosc`: (1) z domyślnymi hiperparametrami, (2) wybrany przez `CrossValidator` na siatce `regParam` x `elasticNetParam` (jak w sekcji 4). Porównaj RMSE OBU modeli na tym samym, wspólnym zbiorze testowym -- czy strojenie faktycznie poprawiło wynik?

<details>
<summary>Pokaż rozwiązanie</summary>

```python
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

tr, wal, te = zamowienia_cechy.randomSplit([0.6, 0.2, 0.2], seed=42)

assembler = VectorAssembler(inputCols=["liczba_pozycji", "srednia_cena"], outputCol="features")
model = LinearRegression(featuresCol="features", labelCol="wartosc")
pipeline = Pipeline(stages=[assembler, model])
ewaluator = RegressionEvaluator(labelCol="wartosc", predictionCol="prediction", metricName="rmse")

# Model 1: domyslne hiperparametry
model_domyslny = pipeline.fit(tr)
rmse_domyslny = ewaluator.evaluate(model_domyslny.transform(te))

# Model 2: wybrany przez CrossValidator
siatka = (
    ParamGridBuilder()
    .addGrid(model.regParam, [0.0, 0.1, 0.5, 1.0])
    .addGrid(model.elasticNetParam, [0.0, 0.5, 1.0])
    .build()
)
cv = CrossValidator(estimator=pipeline, estimatorParamMaps=siatka, evaluator=ewaluator, numFolds=3, seed=42)
cv_model = cv.fit(tr.union(wal.select(tr.columns)))
rmse_strojony = ewaluator.evaluate(cv_model.bestModel.transform(te))

print(f"RMSE (domyslne hiperparametry): {rmse_domyslny:.2f}")
print(f"RMSE (po strojeniu CrossValidator): {rmse_strojony:.2f}")
```

Na tak prostym i malym zestawie cech roznica moze byc niewielka -- strojenie hiperparametrow daje najwieksza korzysc przy wiekszej liczbie cech i wiekszym ryzyku przeuczenia, gdzie regularyzacja ma faktycznie co robic.

</details>

## Co dalej?

W **Module 14** zajmiemy się pracą w samym Databricks Workspace: notatniki, klastry
obliczeniowe, harmonogramowanie zadań (Jobs/Workflows), widgets i `dbutils` -- praktyczna
strona platformy, poza samym kodem Sparka.